![image_1781000778907.png](./image_1781000778907.png "image_1781000778907.png")

![image_1781000798122.png](./image_1781000798122.png "image_1781000798122.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("CustomersOrders").getOrCreate()

# Create customers dataset
customers_data = [
    (1, "Alice Johnson", "alice@example.com"),
    (2, "Bob Martinez", "bob@example.com"),
    (3, "Carol Williams", "carol@example.com"),
    (4, "David Brown", "david@example.com"),
    (5, "Eva Chen", "eva@example.com")
]

customers_columns = ["id", "name", "email"]

customers_df = spark.createDataFrame(customers_data, customers_columns)

# Create orders dataset
orders_data = [
    (1, 1, "2024-01-10"),
    (2, 1, "2024-02-15"),
    (3, 2, "2024-04-20"),
    (4, 2, "2024-05-01"),
    (5, 3, "2023-11-30"),
    (6, 4, "2024-04-28"),
    (7, 5, "2024-01-25"),
    (8, 5, "2024-02-10")
]

orders_columns = ["id", "customer_id", "order_date"]

orders_df = spark.createDataFrame(orders_data, orders_columns)

# Show the DataFrames
print("Customers DataFrame:")
customers_df.show()

print("Orders DataFrame:")
orders_df.show()


In [0]:
recent_date = orders_df.select(f.max(f.to_date('order_date'))).collect()[0][0]

window_spec = Window.partitionBy(customers_df.id).orderBy(f.to_date('order_date').desc())

result_df = (
    customers_df
    .join(orders_df, customers_df.id == orders_df.customer_id)
    .withColumn('rn', f.row_number().over(window_spec))
    .filter(f.col('rn') == 1)
    .select(
        customers_df.id,
        f.col("name"),
        f.to_date('order_date').alias("last_order_date"),
        f.datediff(f.lit(recent_date), f.to_date('order_date')).alias("days_since_last_order")
    )
    .filter(f.col("days_since_last_order") > 90)
)
display(result_df)